# Agriculture & Climate SLM — Retrieval + QLoRA fine-tune

Strategy: this benchmark's reference answers are near-verbatim one-sentence
compressions of a single source document, and `(topic, crop, agro_zone)`
deterministically resolves the correct document for every training row.
So the pipeline is **retrieval (metadata join + TF-IDF fallback) → QLoRA-fine-tuned
a small instruct model that compresses (context, question) into a terse answer**,
not blind generation from parametric knowledge.

Sections:
1. Setup
2. Load data
3. Deterministic + TF-IDF-fallback retrieval (validated against all 45 train rows)
4. Local Levenshtein eval harness
5. Question-paraphrase augmentation (answers untouched, only question surface form varies)
6. QLoRA fine-tune (base model resolved above)
7. Generation + post-processing
8. Assertion-guarded submission.csv write


## 1. Setup

In [ ]:
# Competition runs fully offline (no internet) -- Kaggle's standard GPU image already
# ships peft/bitsandbytes/accelerate/scikit-learn, so this only matters if a package
# is genuinely missing. Never let a failed install kill the whole run.
import subprocess
for pkg in ["peft", "bitsandbytes", "accelerate", "scikit-learn"]:
    try:
        subprocess.run(["pip", "install", "-q", "-U", pkg], timeout=15, check=True,
                        capture_output=True)
    except Exception as e:
        print(f"skip installing {pkg} (offline or already present): {type(e).__name__}")


In [ ]:
import os, re, random, json
import numpy as np
import pandas as pd
import torch
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


## 2. Load data

Auto-discovers the data directory by searching for `train_qa.csv` anywhere under
`/kaggle/input` (works regardless of what the attached dataset or competition is
named, or whether you renamed the folder). Falls back to `/content/` (Colab
uploads) and `./` so this also runs outside Kaggle without edits.

In [ ]:
import glob

def find_data_dir():
    hits = glob.glob("/kaggle/input/**/train_qa.csv", recursive=True)
    if hits:
        return os.path.dirname(hits[0])
    hits = glob.glob("/content/**/train_qa.csv", recursive=True)
    if hits:
        return os.path.dirname(hits[0])
    return "./"

DATA_DIR = find_data_dir()
print("using data dir:", DATA_DIR)

documents = pd.read_csv(os.path.join(DATA_DIR, "documents.csv"))
train_qa  = pd.read_csv(os.path.join(DATA_DIR, "train_qa.csv"))
test_q    = pd.read_csv(os.path.join(DATA_DIR, "test_questions.csv"))

print(documents.shape, train_qa.shape, test_q.shape)


## 3. Retrieval: metadata join + TF-IDF fallback

`(topic, crop, agro_zone)` uniquely resolves the source document for 23/24 documents.
The one ambiguous key (`livestock`/`livestock`/`sub_humid`) and any zero-match
combination at test time fall back to TF-IDF similarity between the question
and candidate document text. Validated below against all 45 training rows —
must be 0 mismatches before trusting it on test.

In [ ]:
key_to_docs = defaultdict(list)
for _, d in documents.iterrows():
    key_to_docs[(d["topic"], d["crop"], d["agro_zone"])].append(d)

def retrieve_doc(topic, crop, agro_zone, question):
    cands = key_to_docs.get((topic, crop, agro_zone), [])
    if len(cands) == 1:
        return cands[0]
    if not cands:
        cands = [d for _, d in documents.iterrows() if d["topic"] == topic]
    if not cands:
        cands = [d for _, d in documents.iterrows()]
    if len(cands) == 1:
        return cands[0]
    texts = [c["title"] + " " + c["text"] for c in cands]
    vec = TfidfVectorizer(ngram_range=(1, 2))
    mat = vec.fit_transform(texts + [question])
    sims = cosine_similarity(mat[-1], mat[:-1])[0]
    return cands[int(sims.argmax())]

# sanity check: must match every training row's labeled document_id
mismatches = 0
for _, row in train_qa.iterrows():
    d = retrieve_doc(row["topic"], row["crop"], row["agro_zone"], row["question"])
    if d["document_id"] != row["document_id"]:
        mismatches += 1
        print("MISMATCH", row["QuestionId"], "expected", row["document_id"], "got", d["document_id"])
assert mismatches == 0, f"{mismatches} retrieval mismatches on training data — fix before proceeding"
print("retrieval validated: 0 mismatches on", len(train_qa), "training rows")


## 4. Local evaluation harness (mean Levenshtein, matches the competition metric)

In [ ]:
# Pure-Python Levenshtein distance -- no external package, works fully offline.
def levenshtein(a, b):
    if a == b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        curr = [i] + [0] * len(b)
        for j, cb in enumerate(b, 1):
            curr[j] = min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + (ca != cb))
        prev = curr
    return prev[-1]

def mean_levenshtein(preds, refs):
    return float(np.mean([levenshtein(p, r) for p, r in zip(preds, refs)]))

def tfidf_baseline_predict(question, topic, train_df=train_qa):
    sub = train_df[train_df["topic"] == topic]
    if sub.empty:
        sub = train_df
    vec = TfidfVectorizer(ngram_range=(1, 2))
    mat = vec.fit_transform(list(sub["question"]) + [question])
    sims = cosine_similarity(mat[-1], mat[:-1])[0]
    return sub.iloc[int(sims.argmax())]["reference_answer"]


## 5. Question-paraphrase augmentation

45 examples is thin for fine-tuning even a narrow behavior. Rather than generating
new (question, answer) pairs from scratch — which risks answers drifting from the
grounding documents — this paraphrases the *question side only* of each existing
training row with the base instruct model and keeps the original, verified
`reference_answer` and `document_id` untouched. This runs once, in-notebook, on a
tiny model, so it stays well inside the committed-run time budget.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Offline run: the base model must come from an attached Kaggle "Models" input, not the Hub.
# Add Input -> Models -> gemma-2 (or your chosen model) in the notebook's Input panel first.
# This scans /kaggle/input for a directory that looks like a HF model (has config.json) and
# whose path mentions "gemma" -- falls back to the Hub string for local/internet-enabled runs.
import glob

def find_local_model(name_hint="gemma"):
    for cfg in glob.glob("/kaggle/input/**/config.json", recursive=True):
        if name_hint in cfg.lower():
            return os.path.dirname(cfg)
    return None

BASE_MODEL = find_local_model("gemma") or "google/gemma-2-2b-it"
print("BASE_MODEL resolved to:", BASE_MODEL)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

gen_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    attn_implementation="eager",  # Gemma-2 logit soft-capping is unstable under sdpa/flash-attn
).to(DEVICE)
gen_model.eval()

PARAPHRASE_PROMPT = (
    "Rewrite the following smallholder farmer's question in a different way, "
    "keeping the same meaning and same topic. Output only the rewritten question, "
    "one line, no quotes, no explanation.\n\nQuestion: {q}\nRewritten:"
)

@torch.no_grad()
def paraphrase(question, n=2, max_new_tokens=32):
    outs = []
    for _ in range(n):
        msgs = [{"role": "user", "content": PARAPHRASE_PROMPT.format(q=question)}]
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
        out = gen_model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=True,
            temperature=0.8, top_p=0.9, pad_token_id=tokenizer.pad_token_id,
        )
        text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        text = text.split("\n")[0].strip().strip('"')
        if text and text.lower() != question.lower() and len(text) > 8:
            outs.append(text)
    return outs

augmented_rows = []
for _, row in train_qa.iterrows():
    for p in paraphrase(row["question"], n=2):
        augmented_rows.append({
            "question": p, "topic": row["topic"], "crop": row["crop"],
            "agro_zone": row["agro_zone"], "document_id": row["document_id"],
            "reference_answer": row["reference_answer"],
        })

augmented_df = pd.DataFrame(augmented_rows)
print("augmented pairs generated:", len(augmented_df))

# free the generation copy before loading the quantized training copy
del gen_model
torch.cuda.empty_cache() if DEVICE == "cuda" else None


## 6. Build the SFT dataset and train/val split

Held-out validation comes only from the *original* 45 rows (never from paraphrases),
since that's the closest proxy to the hidden test distribution.

In [ ]:
PROMPT_TEMPLATE = (
    "Context: {context}\n"
    "Question: {question}\n"
    "Answer (one short sentence):"
)

def build_example(row):
    doc = documents[documents["document_id"] == row["document_id"]].iloc[0]
    prompt = PROMPT_TEMPLATE.format(context=doc["text"], question=row["question"])
    return {"prompt": prompt, "completion": " " + row["reference_answer"].strip()}

train_ids, val_ids = train_test_split(
    train_qa["QuestionId"], test_size=0.2, random_state=SEED
)
train_core = train_qa[train_qa["QuestionId"].isin(train_ids)]
val_core   = train_qa[train_qa["QuestionId"].isin(val_ids)]

# only paraphrases of TRAIN-split rows join the fine-tuning set — val stays unseen
val_core_qids = set(val_core["QuestionId"])
aug_train = augmented_df  # paraphrases were generated from all 45; filter to train-split originals below
train_core_answers = set(train_core["reference_answer"])
aug_train = aug_train[aug_train["reference_answer"].isin(train_core_answers)]

sft_rows = [build_example(r) for _, r in train_core.iterrows()] + \
           [build_example(r) for _, r in aug_train.iterrows()]
print("SFT training examples:", len(sft_rows), "| validation examples:", len(val_core))

from datasets import Dataset
sft_dataset = Dataset.from_list(sft_rows)


## 7. QLoRA fine-tune (base model resolved above -- Gemma-2-2b-it via BASE_MODEL)

Uses plain `transformers.Trainer` instead of `trl.SFTTrainer` -- `trl` is not
reliably present in Kaggle's offline base image, whereas `transformers`, `peft`,
and `bitsandbytes` are. The custom collator below reproduces exactly what
`DataCollatorForCompletionOnlyLM` did: mask the prompt tokens with -100 so the
model is only trained to predict the answer, not to reproduce the context.

In [ ]:
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

# 4-bit quantization (bitsandbytes) is a nice-to-have here, not a requirement -- at 2B
# params a plain bf16 LoRA fine-tune fits comfortably on a single T4/P100, and
# bitsandbytes is not reliably present in an offline Kaggle session. Try it, fall
# back to unquantized bf16 LoRA if it's unavailable rather than failing the run.
quantized = False
try:
    import bitsandbytes  # noqa: F401
    from transformers import BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb_config, device_map="auto",
        attn_implementation="eager",  # Gemma-2 logit soft-capping is unstable under sdpa/flash-attn
    )
    model = prepare_model_for_kbit_training(model)
    quantized = True
    print("using 4-bit quantized LoRA (bitsandbytes available)")
except ImportError:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
        device_map="auto", attn_implementation="eager",
    )
    print("bitsandbytes unavailable -- using plain bf16 LoRA instead of 4-bit QLoRA")

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

MAX_SEQ_LEN = 512

def tokenize_example(example):
    prompt_ids = tokenizer(example["prompt"], add_special_tokens=True)["input_ids"]
    completion_ids = tokenizer(
        example["completion"] + tokenizer.eos_token, add_special_tokens=False
    )["input_ids"]
    input_ids = (prompt_ids + completion_ids)[:MAX_SEQ_LEN]
    labels = ([-100] * len(prompt_ids) + completion_ids)[:MAX_SEQ_LEN]
    return {"input_ids": input_ids, "labels": labels}

tokenized_dataset = sft_dataset.map(tokenize_example, remove_columns=["prompt", "completion"])

def collate_fn(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    pad_id = tokenizer.pad_token_id
    input_ids, labels, attn_mask = [], [], []
    for x in batch:
        n_pad = max_len - len(x["input_ids"])
        input_ids.append(x["input_ids"] + [pad_id] * n_pad)
        labels.append(x["labels"] + [-100] * n_pad)
        attn_mask.append([1] * len(x["input_ids"]) + [0] * n_pad)
    return {
        "input_ids": torch.tensor(input_ids),
        "labels": torch.tensor(labels),
        "attention_mask": torch.tensor(attn_mask),
    }

training_args = TrainingArguments(
    output_dir="/kaggle/working/qlora_out",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="no",
    bf16=(DEVICE == "cuda"),
    seed=SEED,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=collate_fn,
)
trainer.train()


## 8. Generation + post-processing

In [ ]:
model.eval()

def postprocess(text, question):
    text = text.strip()
    text = re.sub(r'^(Answer( \(one short sentence\))?:?\s*)', '', text, flags=re.IGNORECASE)
    text = text.split("\n")[0].strip()
    # keep only the first sentence if the model rambles past one
    m = re.match(r'^(.*?[.!?])(\s|$)', text)
    if m:
        text = m.group(1)
    if text and text[-1] not in ".!?":
        text += "."
    return text.strip() or "See extension guidance for this topic."

@torch.no_grad()
def generate_answer(question, topic, crop, agro_zone, max_new_tokens=32):
    doc = retrieve_doc(topic, crop, agro_zone, question)
    prompt = PROMPT_TEMPLATE.format(context=doc["text"], question=question)
    msgs = [{"role": "user", "content": prompt}]
    chat_prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1,
        pad_token_id=tokenizer.pad_token_id,
    )
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return postprocess(raw, question)


## 9. Validate on the held-out split, compare to the TF-IDF baseline

Sanity check before generating the real submission — fine-tuned model should
beat both the raw TF-IDF baseline and (ideally) a zero-shot RAG prompt on the
same held-out rows.

In [ ]:
val_preds = [
    generate_answer(r["question"], r["topic"], r["crop"], r["agro_zone"])
    for _, r in val_core.iterrows()
]
val_refs = list(val_core["reference_answer"])
print("fine-tuned mean Levenshtein (holdout):", mean_levenshtein(val_preds, val_refs))

baseline_preds = [
    tfidf_baseline_predict(r["question"], r["topic"]) for _, r in val_core.iterrows()
]
print("TF-IDF baseline mean Levenshtein (holdout):", mean_levenshtein(baseline_preds, val_refs))

for p, r in list(zip(val_preds, val_refs))[:5]:
    print("PRED:", p, "\nREF :", r, "\n---")


## 10. Generate test predictions and write the assertion-guarded submission

In [ ]:
rows = []
for _, r in test_q.iterrows():
    ans = generate_answer(r["question"], r["topic"], r["crop"], r["agro_zone"])
    rows.append({"QuestionId": r["QuestionId"], "Answer": ans})

submission = pd.DataFrame(rows)[["QuestionId", "Answer"]]

# hard contract checks — fail loudly rather than submit something broken
assert len(submission) == len(test_q), "row count mismatch"
assert list(submission["QuestionId"]) == list(test_q["QuestionId"]), "QuestionId order mismatch"
assert submission["Answer"].str.len().gt(0).all(), "empty answer present"
assert submission["Answer"].notna().all(), "null answer present"

submission.to_csv("/kaggle/working/submission.csv", index=False)
print("wrote /kaggle/working/submission.csv")
submission
